# Small vs frontier: quality vs cost

**Session 7 · Track A · local Ollama**

Run one task on a local and a hosted model; compare quality, cost, latency.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
# switch MODEL_BACKEND between 'ollama' and 'openai' and re-run
from eval import load_cases, run_eval, print_report, exact


### Worked example

The same classifier task run on a small local model and a frontier model, with accuracy and latency logged. Missing model/key is skipped, not fatal.


In [ ]:
# Worked example: same task, two models, numbers side by side
import time, os
from eval import load_cases, run_eval, exact
import ollama
from openai import OpenAI

cases = load_cases("../eval/datasets/sentiment.jsonl")
LABS = ["positive", "negative", "neutral"]
PROMPT = 'Classify sentiment as positive, negative, or neutral. One lowercase word.\nText: "{t}" ->'

def ollama_call(text):
    r = ollama.chat(model="llama3.2", options={"temperature": 0},
                    messages=[{"role": "user", "content": PROMPT.format(t=text)}])
    return r["message"]["content"]

def openai_call(text):
    r = OpenAI().chat.completions.create(
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"), temperature=0,
        messages=[{"role": "user", "content": PROMPT.format(t=text)}])
    return r.choices[0].message.content

def make(caller):
    def f(text):
        out = caller(text).strip().lower()
        return next((l for l in LABS if l in out), out)
    return f

for name, caller in [("local  llama3.2", ollama_call), ("frontier gpt-4o-mini", openai_call)]:
    t0 = time.time()
    try:
        rep = run_eval(cases, make(caller), scorer=exact)
        print(f"  {name:22} acc={rep['accuracy']:.0%}  {time.time() - t0:.1f}s")
    except Exception as e:
        print(f"  {name:22} skipped: {type(e).__name__} - configure that model/key to include it")


## Your turn - vary the example

1. Point `ollama_call` at a smaller model (`qwen2.5:3b`). How far does accuracy fall?
2. Add token counts to the log and turn latency + price into a cost-per-1k-calls number.
3. Recommend one model + prompt for this task, with the numbers that justify it.


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
